# 마크다운 전처리 → 지정 폴더에 `.md` 저장

- 로드: `md_pages`(또는 이미 메모리에 있는 `docs`)
- 제거: 사이트 공통 푸터(`Heum` + 주소 블록), Greeting 문구, 끝의 `TOP`, 단독 `![](...)` 이미지 줄, 과한 `*`
- 저장: 원본 파일명(`metadata['source']`)을 유지해 `OUTPUT_DIR`에 기록

경로는 아래 셀에서 수정하세요.

In [7]:
from __future__ import annotations

import re
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document

# 노트북 실행 시 보통 notebooks/ 가 cwd — 필요하면 절대 경로로 바꿉니다.
NOTEBOOK_DIR = Path.cwd()
INPUT_MD_DIR = NOTEBOOK_DIR / "md_pages"
OUTPUT_DIR = NOTEBOOK_DIR / "md_pages_clean"

GLOB = "**/*.md"

In [8]:
def extract_page_url(text: str) -> str | None:
    """본문 첫 줄이 `# https://...` 형태일 때 URL만 추출합니다."""
    m = re.match(r"^#\s+(https?://\S+)\s*", text.strip(), re.MULTILINE)
    return m.group(1) if m else None


def preprocess_markdown(text: str) -> str:
    """
    크롤된 마크다운에서 RAG에 불리한 반복 구역(푸터·배너 이미지 줄 등)을 제거·정리합니다.
    사이트 구조가 바뀌면 푸터 정규식을 조정해야 할 수 있습니다.
    """
    t = text.replace("\r\n", "\n")

    # 푸터: "Heum" 다음 줄 "위치 :" 로 시작하는 블록부터 끝까지(연락처·링크 리스트 포함)
    t = re.sub(r"\nHeum\n위치\s*:[\s\S]*$", "\n", t, flags=re.MULTILINE)

    # Greeting 제작 문구가 남은 경우
    t = re.sub(r"\n\nmade with \[Greeting\][\s\S]*$", "\n", t)

    # 페이지 하단 "TOP" 플로팅 등
    t = re.sub(r"\nTOP\s*$", "", t)

    # 이미지만 있는 마크다운 줄 제거 (본문 의미가 거의 없고 URL만 길게 잡아먹음)
    t = re.sub(r"^\s*!\[[^\]]*\]\([^)]+\)\s*$", "", t, flags=re.MULTILINE)

    # 깨진/중복 볼드 마크다운 완화
    t = re.sub(r"\*{4,}", "**", t)

    # 빈 줄 과다 축소
    t = re.sub(r"\n{3,}", "\n\n", t)

    return t.strip()


def save_preprocessed_documents(
    documents: list[Document],
    output_dir: Path,
    *,
    preprocess_fn=preprocess_markdown,
) -> list[Path]:
    """
    LangChain Document 리스트를 전처리한 뒤 output_dir에 .md 파일로 저장합니다.
    파일명은 metadata['source']의 basename을 사용합니다.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written: list[Path] = []
    for doc in documents:
        src = doc.metadata.get("source") or ""
        name = Path(str(src)).name if src else f"doc_{abs(hash(doc.page_content))}.md"
        if not name.lower().endswith(".md"):
            name = f"{name}.md"
        body = preprocess_fn(doc.page_content)
        out_path = output_dir / name
        out_path.write_text(body, encoding="utf-8")
        written.append(out_path)
    return written

In [9]:
def load_md_documents(md_dir: Path, glob_pattern: str = GLOB) -> list[Document]:
    """디렉터리에서 .md를 LangChain Document로 로드합니다."""
    md_dir = Path(md_dir)
    if not md_dir.is_dir():
        raise FileNotFoundError(f"디렉터리가 없습니다: {md_dir}")
    loader = DirectoryLoader(
        str(md_dir),
        glob=glob_pattern,
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    return loader.load()


# 이미 상위 셀에서 docs를 만들었다면 이 블록은 건너뛰어도 됩니다.
try:
    docs  # noqa: B018
except NameError:
    docs = load_md_documents(INPUT_MD_DIR)

len(docs), docs[0].metadata.get("source") if docs else None

(31,
 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai.md')

In [10]:
saved_paths = save_preprocessed_documents(docs, OUTPUT_DIR)
print(f"저장 완료: {len(saved_paths)}개 → {OUTPUT_DIR.resolve()}")
saved_paths[:3]

저장 완료: 31개 → C:\Users\jw160\project\RAG\notebooks\md_pages_clean


[WindowsPath('c:/Users/jw160/project/RAG/notebooks/md_pages_clean/https___www_heum_ai.md'),
 WindowsPath('c:/Users/jw160/project/RAG/notebooks/md_pages_clean/https___www_heum_ai_apply.md'),
 WindowsPath('c:/Users/jw160/project/RAG/notebooks/md_pages_clean/https___www_heum_ai_apply_faq.md')]